# CIFAR-10 image classification with TensorFlow

Practice activity from the Microsoft Foundations of AI and Machine Learning course on Coursera. The goal was to build a small CNN from scratch, train it on CIFAR-10, evaluate it, and save it so I could reload it later.

I ran this on an Azure ML compute instance (Standard_E4ds_v4, CPU only), which is why I capped training at 3 epochs. With a GPU you could comfortably push to 10 or more epochs for better accuracy.

## 1. Imports

In [1]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt

print('TensorFlow version:', tf.__version__)

2026-05-11 18:12:37.395739: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-11 18:12:45.191746: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-11 18:12:55.125274: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow version: 2.20.0


## 2. Load and normalize CIFAR-10

TensorFlow ships with CIFAR-10 built in so no manual download. Pixel values get divided by 255 to land them in the 0 to 1 range, which makes training a lot smoother.

In [2]:
(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()

# Normalize pixels to [0, 1]
x_train, x_test = x_train / 255.0, x_test / 255.0

print('Training data shape:', x_train.shape)
print('Test data shape:', x_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Training data shape: (50000, 32, 32, 3)
Test data shape: (10000, 32, 32, 3)


## 3. Build the CNN

Three convolution blocks, each with a 3x3 filter and ReLU, with max pooling between the first two pairs. Then flatten and feed into a dense hidden layer plus a 10 unit output (one per class). Logits stay raw since I use SparseCategoricalCrossentropy with from_logits=True in the compile step.

In [5]:
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(10))

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 30, 30, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 15, 15, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 13, 13, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 4, 4, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        65,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 122,570 (478.79 KB)

 Trainable params: 122,570 (478.79 KB)

 Non-trainable params: 0 (0.00 B)

## 4. Train the model

Adam optimizer with the standard cross entropy loss for multiclass classification. Only 3 epochs because this is running on CPU and each epoch takes around 15 to 20 seconds. With more compute or more epochs I would expect to climb above 70 percent test accuracy.

In [6]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

history = model.fit(x_train, y_train, epochs=3, validation_data=(x_test, y_test))

Epoch 1/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - accuracy: 0.4413 - loss: 1.5293 - val_accuracy: 0.5406 - val_loss: 1.2803
Epoch 2/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - accuracy: 0.5855 - loss: 1.1641 - val_accuracy: 0.6150 - val_loss: 1.1041
Epoch 3/3
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - accuracy: 0.6457 - loss: 1.0092 - val_accuracy: 0.6541 - val_loss: 0.9922


## 5. Evaluate on the test set

In [7]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f'\nTest accuracy: {test_acc:.4f}')
print(f'Test loss: {test_loss:.4f}')

313/313 - 1s - 3ms/step - accuracy: 0.6541 - loss: 0.9922

Test accuracy: 0.6541
Test loss: 0.9922


## 6. Save and reload the model

Sanity check that saving and reloading produces the same numbers. I used the new Keras format (.keras) since the older .h5 path prints a deprecation warning.

In [8]:
model.save('my_cifar10_model.keras')
print('Model saved as my_cifar10_model.keras')

loaded_model = tf.keras.models.load_model('my_cifar10_model.keras')
print('Model reloaded successfully')

loaded_test_loss, loaded_test_acc = loaded_model.evaluate(x_test, y_test, verbose=2)
print(f'Reloaded model test accuracy: {loaded_test_acc:.4f}')

Model saved as my_cifar10_model.keras
Model reloaded successfully
313/313 - 1s - 4ms/step - accuracy: 0.6541 - loss: 0.9922
Reloaded model test accuracy: 0.6541


## Reflection

A few things that stood out while doing this exercise:

1. **Kernel selection matters.** My first run used the default Python 3.10 SDK v2 kernel which did not have TensorFlow installed. Pip installing it inside the notebook caused dependency conflicts with the preinstalled keras and jupyterlab versions. Switching to the prebuilt Python 3.10 Pytorch and Tensorflow kernel solved it cleanly.
2. **CPU training is slow.** 3 epochs on CIFAR-10 took about 52 seconds total on this Standard_E4ds_v4 instance. A GPU would make 10 or more epochs trivial. Next time I will pick GPU compute when working with image data.
3. **Normalizing pixels helps a lot.** Dividing by 255 noticeably stabilizes training even on a tiny network like this one.
4. **Saving as .keras is now preferred.** The older .h5 example from the course material still works but Keras emits a deprecation note.

Final test accuracy after 3 epochs landed around 65 percent. Not great, but in line with what a small CNN can do on CIFAR-10 without augmentation or extra training. Next thing I want to try is adding dropout and image augmentation to push past 75 percent.